# AI-OV7 scale-up — one box, four GPUs

Ported from `notebooks/kaggle_generate_pairs.ipynb` on `feat/ai-ov7-generation`,
which is the notebook that produced the first pairs. **The logic is not ported
with it, and that is the point.** That notebook carried `crop_box`, the encoder
copy, the shard deal and the generation loop inline, and `docs/ai_ov7_generation.md`
§3 records what inline cost: 6 of 6 smoke pairs leaked the label, through two
bugs in exactly those functions — fakes always a multiple of 8 while reals kept
their native size, and a DCT phase mismatch that read as `jpeg_quality` AUC
0.0000. Both were in pure functions a test would have caught, and both now live
in `src/aigcdet/generate/` under 104 tests.

So every cell below **drives the tested code** rather than restating it. If you
find yourself pasting a `crop_box` in here, that is the bug coming back.

## What this run is

`docs/02` U6: the corpus is 11,978 pairs over nine families and five decoder
lineages. This takes it to ~42,000 by adding lineage *breadth*, because §11
measured breadth as the thing that moved the gate — `laplacian_var` 0.5998 →
0.5632 from the supplement alone, on no extra volume.

| shard | order positions | suite | state |
|---|---|---|---|
| 0 | 0 – 10,924 | `ov7` | 9,978 used, ~925 free |
| 1 | 10,925 – 21,849 | `ov7_lineage` | 2,000 used, ~8,925 free |
| 2 | 21,850 – 32,774 | `ov7_lineage2` | free |
| 3 | 32,775 – 43,699 | `ov7_lineage3` | free |
| 4 | 43,700 – 54,623 | `ov7` or `ov7_lineage` | free |

Shards 2–4 are unclaimed, so which suite takes which is a live choice — the
table is the current plan, not a fact on disk. `ov7_lineage3` is one family,
`zimage_t2i`, and it is the only arm in the corpus whose lineage claim is not
backed by a measurement: it puts `flux1_vae` into training while `flux2_vae` is
the held-out rung, and nobody has measured whether those two decoders are the
same. `validate_suite` warns about that every time it runs. Retire the warning
with `features/recon.py` on the two VAEs; until then, report any held-out
number with the caveat attached.

**Stay on `--n-shards 5`.** Re-gridding to 4 puts a boundary at 13,656, inside
shard 1's block, so a run there re-deals reals the supplement already generated:
a share dict *is* the deal, and `_done_ids` is per family, so one scene would
land twice on the generated side against one real. `generate_ov7.used_elsewhere`
refuses that now — treat it as the backstop, not the plan.

## The three rules

1. **Do not change `--seed`, or any suite's shares.** The strata pattern is a
   function of the share dict. A different one re-deals every real in the block.
2. **A suite may only grow on a shard it already owns, or on a fresh one.**
3. **Run the gate before anything trains.** `docs/02` §5 says it is allowed to
   cancel the task, and §6 says a failure is a finding to write up.

---

# READ THIS FIRST

**What this makes.** For each real photograph, one AI counterpart of the same
scene, at the same pixel size, written through the same JPEG quantisation
tables. Content, geometry and encoder are held fixed, so what is left between a
pair is the generator. One real : one fake, and no real is used twice — across
suites as well as within one.

## Order of operations

1. Run sections 0–3. Section 3 is a hard gate: **`jpegtran` missing kills all
   four workers before a single model loads.**
2. Section 4 stages the reals, pool and captions. **Captions are precomputed
   here and never during generation** — `caption_pool` rewrites the whole
   parquet on every log tick, so four workers on one path lose captions to
   last-writer-wins.
3. Section 6, `SMOKE = True`. Look at section 7. A fake that is a different
   scene, or identical to its real, stops the run.
4. Section 8, the real run. ~3 h across four GPUs.
5. **Section 9, the gate.** Then section 10 freezes the outputs.

## Splitting work across GPUs

Two different axes, and using the wrong one silently corrupts the deal.

* **`--shard`** splits the *reals* across identical work. Shard *k* owns block
  *k*; blocks are disjoint.
* **`--run-families`** splits the *families of one shard's deal* across boxes.
  Every process passes an identical `--suite/--total/--families`, so the strata
  pattern is the same on all of them, and this only says which of the dealt
  families this process executes. It is what lets one shard span heterogeneous
  hardware.
* **`--families`** is neither. It **changes the deal** — shares are
  renormalised over what is left — and is for smoke runs only.

## 0. Get the code

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"

ON_KAGGLE = os.path.isdir("/kaggle/input")
HERE      = Path("/kaggle/working" if ON_KAGGLE else os.environ.get("OV7_ROOT", "/workspace"))
REPO_DIR  = HERE / "robust-aigc-detection"
HERE.mkdir(parents=True, exist_ok=True)

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv), flush=True)
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if (REPO_DIR / ".git").is_dir():
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

# `src` on the path, NOT `pip install -e .` -- that hands pip the torch>=2.0
# line and invites it to replace a torch matched to this box's drivers.
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
sh(["git", "log", "--oneline", "-1"])

## 1. Parameters

`SHARD` and `SUITE` are the two that matter, and section 2 checks them against
the shard table above rather than trusting you.

In [ ]:
# ================= THE ONLY LINES YOU NORMALLY EDIT =================
SMOKE   = True          # 2 per family into a throwaway tree. Do this first.
SHARD   = 2             # 2, 3 or 4 are free. 0 and 1 are RESUMES -- see §8.
SUITE   = "ov7_lineage2"   # 2 -> ov7_lineage2, 3 -> ov7_lineage3, 4 -> ov7/ov7_lineage
TOTAL   = 10000         # generated images for this shard
RUN_FAMILIES = None     # None = every family this shard was dealt.
                        # A list splits ONE shard's families across boxes:
                        # the 24 GB card takes the heavy model, another takes
                        # the rest. Both pass the same SUITE/TOTAL.
# ====================================================================

N_SHARDS = 5            # NEVER change this. Re-gridding re-deals shard 1.
SEED     = 20260830     # NEVER change this. It IS the deal.

# Staged inputs. Section 4 puts them here.
PORTRAIT  = HERE / "portrait"                  # 60,000 reals, ~5 GB
POOL      = HERE / "ov7_pool.parquet"
CAPTIONS  = HERE / "ov7_captions.parquet"      # PRECOMPUTED and merged
ATTRIB    = HERE / "attribution.csv"

# Outputs. `--out` is shared across workers and that is safe -- image ids are
# disjoint across shards, so no two workers write the same file. `--rows-dir`
# is NOT: run_family appends one jsonl per family with prompts of unbounded
# length, and four processes interleaving those writes corrupt lines.
OUT_ROOT  = HERE / ("smoke_ov7" if SMOKE else "raw_ov7_src")
ROWS_DIR  = HERE / f"_rows_{SHARD}"
HF_HOME   = HERE / "hf"          # SHARED, or each worker pulls the same weights

N_GPUS    = 4
for d in (OUT_ROOT, ROWS_DIR, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print(f"repo    {REPO_DIR}\nshard   {SHARD}/{N_SHARDS}  suite {SUITE}\n"
      f"out     {OUT_ROOT}\nrows    {ROWS_DIR}")

## 2. What this shard is allowed to do

The shard/suite pairing is not free choice: a suite may only grow on a shard it
already owns, or on a fresh one. This cell refuses the combinations that would
re-deal reals that already have fakes — the same thing
`generate_ov7.used_elsewhere` refuses at run time, checked here before you have
staged 5 GB.

In [ ]:
from aigcdet.generate import registry

OWNED = {0: "ov7", 1: "ov7_lineage"}      # docs/02 U5, as generated
FREE  = {2, 3, 4}

assert N_SHARDS == 5, "docs/02 U5: re-gridding puts a boundary inside shard 1"
assert SUITE in registry.SUITES, f"unknown suite {SUITE!r}: {sorted(registry.SUITES)}"
if SHARD in OWNED:
    assert SUITE == OWNED[SHARD], (
        f"shard {SHARD} was generated with {OWNED[SHARD]!r}. Running {SUITE!r} "
        f"on it re-deals every real in the block: a real that is sdxl_t2i "
        f"today comes out something else tomorrow, and _done_ids is per family "
        f"so the second fake is simply generated.")
    print(f"shard {SHARD}: RESUME of {SUITE!r}. Stage the existing _rows/*.jsonl "
          f"and output tree first (§8) or this regenerates from scratch.")
else:
    assert SHARD in FREE, f"shard {SHARD} is not in the 0-4 grid"
    print(f"shard {SHARD}: fresh, {SUITE!r} may claim it")

suite  = registry.SUITES[SUITE]
corpus = registry.corpus_of(SUITE)

import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    registry.validate_suite(suite, corpus=corpus)
for w in caught:
    print(f"WARNING: {w.message}\n")

print(f"\n{'family':22s} {'model':18s} {'lineage':14s} {'method':10s} {'share':>6s}")
for name, fam in sorted(suite.items()):
    spec = registry.MODELS[fam.model]
    print(f"{name:22s} {fam.model:18s} {spec.lineage:14s} {fam.method:10s} "
          f"{fam.share:6.2f}")

held = registry.heldout_groups(corpus)[0]
trained = sorted({registry.lineage_of(f, corpus) for f in corpus} - {registry.HELDOUT_LINEAGE})
print(f"\ncorpus after this run: {len(corpus)} families, "
      f"{len(trained) + 1} lineages")
print(f"  held out ({registry.HELDOUT_LINEAGE}): {held}")
print(f"  trained: {trained}")

## 3. Preflight — every one of these kills the run, not just this cell

`jpegtran` first because it is the one that is not obvious. It is a hard
`ap.error` at startup, not a soft dependency: it is the only way to crop a real
without re-encoding it, and re-encoding the real would add a compression
generation to the authentic class — manufacturing the exact confound the qtable
copy exists to remove.

In [ ]:
import torch

problems, notes = [], []

if shutil.which("jpegtran") is None:
    problems.append("jpegtran is not on PATH. `apt-get install -y "
                    "libjpeg-turbo-progs`. Without it all four workers die "
                    "before a single model loads.")

if not torch.cuda.is_available():
    problems.append("no CUDA device")
else:
    n = torch.cuda.device_count()
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        bf16 = p.major >= 8
        notes.append(f"  cuda:{i}  {p.name}  sm_{p.major}{p.minor}  "
                     f"{p.total_memory / 2**30:.0f} GB  bf16={'yes' if bf16 else 'NO'}")
        if not bf16:
            problems.append(
                f"cuda:{i} ({p.name}) is sm_{p.major}{p.minor}: no bf16. Not a "
                f"speed issue -- fp16 has 5 exponent bits against bf16's 8 and "
                f"these models use the range. It does not raise; it emits "
                f"washed-out or NaN-speckled images that still look like "
                f"images, which for a corpus recording what a generator's "
                f"output LOOKS LIKE is the worst available failure.")
    if n < N_GPUS:
        notes.append(f"  only {n} GPUs visible, N_GPUS={N_GPUS} -- section 8 "
                     f"will launch {n}")

free_gb = shutil.disk_usage(HERE).free / 2**30
if free_gb < 60:
    problems.append(f"{free_gb:.0f} GB free at {HERE}. Need ~5 GB reals + "
                    f"~33 GB weights + ~10 GB output.")

vram = max((torch.cuda.get_device_properties(i).total_memory / 2**30
            for i in range(torch.cuda.device_count())), default=0)
for name, fam in suite.items():
    spec = registry.MODELS[fam.model]
    if spec.vram_gb > vram:
        notes.append(f"  {name}: {spec.vram_gb} GB weights on a {vram:.0f} GB "
                     f"card -> {spec.offload_mode} CPU offload")

print("\n".join(notes) or "(no devices)")
print(f"\ndisk free at {HERE}: {free_gb:.0f} GB   HF_HOME: {HF_HOME}")
if problems:
    for p_ in problems:
        print("\nPROBLEM:", p_)
    raise SystemExit("fix the above before staging anything")
print("\npreflight ok")

## 4. Stage the inputs

Four things, and the caption file is the one with a trap in it.

`caption_pool` rewrites the **whole** parquet on every log tick. Four workers
pointed at one path race, and the last writer wins — each flush carries only
that worker's captions and silently discards the others'. The reals whose
captions were lost then generate on an empty prompt, which `run_family` refuses,
so it surfaces as a runaway failure rate in a family rather than as anything
about captions.

So captions are computed **here**, in parts, merged once, and the merged file is
only ever read afterwards. `scripts/caption_ov7.py` is that.

In [ ]:
# Point these at wherever the reals actually are on this box.
SRC_PORTRAIT = os.environ.get("OV7_SRC_PORTRAIT", "/mnt/berstorage/techjam/open_images/portrait")
SRC_ATTRIB   = os.environ.get("OV7_SRC_ATTRIB",   "/mnt/berstorage/techjam/open_images/attribution.csv")

if not PORTRAIT.exists() and Path(SRC_PORTRAIT).exists():
    print(f"staging reals {SRC_PORTRAIT} -> {PORTRAIT} (~5 GB)")
    shutil.copytree(SRC_PORTRAIT, PORTRAIT)
if not ATTRIB.exists() and Path(SRC_ATTRIB).exists():
    shutil.copy(SRC_ATTRIB, ATTRIB)

n_reals = len(list(PORTRAIT.glob("*.jpg"))) if PORTRAIT.exists() else 0
print(f"reals: {n_reals} at {PORTRAIT}")
assert n_reals, (f"no reals at {PORTRAIT}. Stage them, or set OV7_SRC_PORTRAIT. "
                 f"CC BY 2.0 only -- do NOT substitute Pexels or Unsplash "
                 f"(docs/02 §1).")
assert ATTRIB.exists(), (f"no attribution.csv. CC BY REQUIRES attribution and "
                         f"normalisation strips image metadata, so this file is "
                         f"the only surviving record. docs/02 §5.4 gates on it.")

In [ ]:
# The pool: a header-only probe of every real, cached. ~9% come out
# ineligible on encoder reproducibility, itemised in ai_ov7_generation.md §2.
import pandas as pd
from aigcdet.generate.pool import build_pool, rebase_paths

if POOL.exists():
    pool = rebase_paths(pd.read_parquet(POOL), PORTRAIT)
else:
    t0 = time.time()
    pool = build_pool(PORTRAIT, ATTRIB)
    pool.to_parquet(POOL, index=False)
    print(f"probed in {time.time() - t0:.0f}s")

elig = int(pool.eligible.sum())
print(f"pool: {elig} eligible of {len(pool)} ({elig / len(pool):.1%})")
print(pool.loc[~pool.eligible, "reason"].str.slice(0, 46).value_counts().head().to_string())

In [ ]:
# Captions, in parts, then merged. ~0.045 s/image, so the whole eligible pool
# is ~35 min on one GPU and ~9 across four. Skipped entirely if the merged file
# is already there -- a caption is an input to seed-deterministic generation,
# so REGENERATING CAPTIONS REGENERATES THE CORPUS.
if CAPTIONS.exists():
    have = pd.read_parquet(CAPTIONS)
    print(f"captions already merged: {len(have)} at {CAPTIONS} -- not recomputed")
else:
    parts = [HERE / f"_caps_{k}.parquet" for k in range(N_GPUS)]
    procs = []
    for k, part in enumerate(parts):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), HF_HOME=str(HF_HOME))
        argv = [sys.executable, "scripts/caption_ov7.py",
                "--pool", str(POOL), "--portrait-dir", str(PORTRAIT),
                "--part", str(k), "--n-parts", str(N_GPUS), "--out", str(part)]
        if SMOKE:
            argv += ["--limit", "40"]
        print("$", " ".join(argv), flush=True)
        procs.append(subprocess.Popen(argv, env=env))
    rc = [p_.wait() for p_ in procs]
    assert not any(rc), f"a caption worker failed: exits {rc}"
    sh([sys.executable, "scripts/caption_ov7.py", "--merge",
        *[str(x) for x in parts], "--out", str(CAPTIONS)])

## 5. What this box will actually run

`resolve_suite` divides `TOTAL` by the suite's shares with largest-remainder, so
the counts sum to exactly `TOTAL` and no family rounds to zero. `select` then
deals reals to families as a repeating stratum pattern over the shard's block —
not a contiguous slice per family, which is what makes `--total 3000` a strict
prefix of `--total 10000` instead of a reshuffle.

In [ ]:
from aigcdet.generate.pool import select

counts = registry.resolve_suite(TOTAL, suite, corpus=corpus)
shares = {k: v.share for k, v in suite.items()}
sel = select(pool, counts, seed=SEED, shard=SHARD, n_shards=N_SHARDS, shares=shares)

if RUN_FAMILIES:
    sel = sel.loc[sel["family"].isin(RUN_FAMILIES)].reset_index(drop=True)

print(f"{sum(counts.values())} reals dealt across {len(counts)} families, "
      f"shard {SHARD}/{N_SHARDS}\n")
print(sel.groupby("family").size().to_string())
print(f"\nfirst real in this block: {sel['image_id'].iloc[0]}")
print(f"reals used twice anywhere: {int(sel['image_id'].duplicated().sum())}")

## 6. Smoke — prove the chain before spending the GPU

Two per family, into a throwaway tree. Everything the real run does happens
here: the licence check against the Hub, the load, the size rounding, the
degenerate-output guards, the encoder copy, and `assert_parity` on every pair.

`--families` is used here and only here: it renormalises shares, so a smoke run
deliberately does **not** reproduce the real deal.

In [ ]:
if not SMOKE:
    print("SMOKE is False -- skipping. Set it True if you have not smoked today.")
else:
    smoke_out = HERE / "smoke_ov7"
    argv = [sys.executable, "scripts/generate_ov7.py",
            "--suite", SUITE, "--total", str(2 * len(suite)), "--smoke",
            "--shard", str(SHARD), "--n-shards", str(N_SHARDS),
            "--pool", str(POOL), "--portrait-dir", str(PORTRAIT),
            "--captions", str(CAPTIONS),
            "--out", str(smoke_out), "--rows-dir", str(smoke_out / "_rows")]
    t0 = time.time()
    rc = subprocess.run(argv, env=dict(os.environ, HF_HOME=str(HF_HOME))).returncode
    print(f"\nexit {rc} after {time.time() - t0:.0f}s")
    assert rc == 0, ("SMOKE FAILED. Read the traceback, then the playbook at "
                     "the bottom. Do NOT start the real run.")

## 7. Look at them — the step that is not a number

`docs/ai_ov7_generation.md` §11 caught two failures here that no aggregate
statistic would have shown: Kandinsky returning **the same image twice** for two
different requested sizes, and Sana generating a 432×640 portrait request at
1216×832 landscape and squashing it down. Both produced perfectly plausible
rows.

What you are checking, per family:

* the fake is the **same scene** as its real — if it is an unrelated photo, the
  prompt did not reach the model;
* it is **visibly redrawn**, not a copy — a copy trains a detector to call
  photographs fake, and `run.check` rejects at Δ < 1.5, which is a floor and not
  a standard;
* it is not washed out or NaN-speckled, which is what fp16 on a bf16 model looks
  like.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

rows_root = (HERE / "smoke_ov7" / "_rows") if SMOKE else ROWS_DIR
seen = {}
for jl in sorted(Path(rows_root).glob("rows_*.jsonl")):
    for line in jl.read_text().splitlines():
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        seen.setdefault(r["family"], r)

root = (HERE / "smoke_ov7") if SMOKE else OUT_ROOT
if not seen:
    print(f"no rows under {rows_root} yet")
else:
    fams = sorted(seen)
    fig, ax = plt.subplots(2, len(fams), figsize=(3.2 * len(fams), 7.0),
                           squeeze=False)
    for j, fam in enumerate(fams):
        r = seen[fam]
        ax[0][j].imshow(Image.open(root / r["real_rel"]))
        ax[0][j].set_title("real", fontsize=9)
        ax[1][j].imshow(Image.open(root / r["fake_rel"]))
        ax[1][j].set_title(f"{fam}\n{r['width']}x{r['height']}  "
                           f"{r.get('gen_seconds', 0):.1f}s", fontsize=8)
        ax[0][j].axis("off"); ax[1][j].axis("off")
    plt.tight_layout(); plt.show()
    for fam in fams:
        r = seen[fam]
        print(f"{fam:22s} q={r['jpeg_quality']:.0f} {r['gpu']} "
              f"{r['dtype']}\n    {r['prompt'][:96]!r}")

## 8. The real run — one process per GPU

Set `SMOKE = False` in section 1 and re-run sections 1–5 first.

**Shared `--out` is safe**: image ids are disjoint across shards, so no two
workers write the same file. **Separate `--rows-dir` is not optional**:
`run_family` appends one jsonl per family with prompts of unbounded length, and
four processes interleaving those writes corrupt lines. They are concatenated in
section 10.

This cell splits **one** shard's families across the box's GPUs with
`--run-families`. Every process passes an identical `--suite`, `--total` and
`--shard`, so all four compute the same deal and each executes a slice of it.

**Resuming shard 0 or 1** needs the existing `_rows/*.jsonl` **and** the output
tree staged first: `_done_ids` checks the jsonl *and* both files on disk, so it
skips what exists and generates only the tail. Without them it regenerates from
scratch into the same paths.

In [ ]:
fams = sorted(RUN_FAMILIES or suite)
n_gpu = min(N_GPUS, torch.cuda.device_count())
# Model-major, so a family sharing weights with another lands on one GPU and
# the 33 GB of weights is not pulled four times.
lanes = [fams[i::n_gpu] for i in range(n_gpu)]
lanes = [l for l in lanes if l]

if SMOKE:
    print("SMOKE is still True. Set SMOKE = False in section 1, re-run 1-5, "
          "then this cell.")
else:
    procs = []
    for k, lane in enumerate(lanes):
        rows_k = HERE / f"_rows_{SHARD}_gpu{k}"
        rows_k.mkdir(parents=True, exist_ok=True)
        argv = [sys.executable, "scripts/generate_ov7.py",
                "--suite", SUITE, "--total", str(TOTAL),
                "--shard", str(SHARD), "--n-shards", str(N_SHARDS),
                "--seed", str(SEED),
                "--pool", str(POOL), "--portrait-dir", str(PORTRAIT),
                "--captions", str(CAPTIONS),
                "--out", str(OUT_ROOT), "--rows-dir", str(rows_k),
                "--run-families", ",".join(lane)]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), HF_HOME=str(HF_HOME))
        print(f"[gpu {k}] {lane}")
        print("  $", " ".join(argv), flush=True)
        procs.append((k, rows_k, subprocess.Popen(argv, env=env)))

    t0 = time.time()
    rc = [(k, p_.wait()) for k, _, p_ in procs]
    print(f"\nall workers exited after {(time.time() - t0) / 3600:.2f} h: {rc}")
    for k, code_ in rc:
        if code_:
            print(f"  gpu {k} exit {code_} -- see its stats.json, and the "
                  f"playbook at the bottom. A nonzero exit can mean 'some rows "
                  f"failed', not 'nothing ran': re-running resumes.")

## 9. The gate — run it before anything trains

`docs/02` §5: this is **allowed to cancel the task**. §6: a failure that encoder
parity does not fix is a real finding, not something to work around.

The comparison is against `ai_ov7_generation.md` §10's nine-family reading, not
against the frozen corpus — the corpus baselines are the thing this source is
supposed to be *better* than.

| proxy | 9 families | frozen corpus |
|---|---|---|
| `jpeg_quality` | 0.5152 | 0.5532 |
| `laplacian_var` | 0.5632 | 0.6721 |
| `noise_floor`  | 0.5072 | 0.6374 |
| `short_side`   | 0.5015 | 0.5992 |

**If `jpeg_quality` alone is above ~0.60, fix the save path and look at nothing
else** (§5.2). Everything else is a property of the generators.

In [ ]:
gate_out = HERE / f"gate_shard{SHARD}.json"
argv = [sys.executable, "scripts/gate_confounds.py", "--n", "4000"]
print("$", " ".join(argv))
print("\nIf this script takes different arguments on your branch, read its "
      "--help: the point is that the SAME estimators that produced\n"
      "0.5152 / 0.5632 / 0.5072 / 0.5015 produce these numbers. A "
      "re-implementation that is merely equivalent compares two estimators.")
subprocess.run(argv)

## 10. Freeze — merge the rows, keep the attribution

In [ ]:
import pandas as pd

rows = []
for rd in sorted(HERE.glob(f"_rows_{SHARD}*")):
    for jl in sorted(rd.glob("rows_*.jsonl")):
        for line in jl.read_text().splitlines():
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass          # truncated final line of a kill

if not rows:
    print("nothing generated yet")
else:
    df = pd.DataFrame(rows).drop_duplicates(["family", "image_id"], keep="last")
    pairs = OUT_ROOT / f"pairs_shard{SHARD}.parquet"
    df.to_parquet(pairs, index=False)

    dup = df["image_id"].duplicated().sum()
    print(df.groupby(["lineage", "family"]).size().to_string())
    print(f"\n{len(df)} pairs -> {pairs}")
    print(f"reals used by more than one family: {dup}   (must be 0)")
    assert dup == 0, ("a real has fakes from two families. One real produces "
                      "one fake for one family; this puts a scene in the "
                      "corpus twice on the generated side against one real.")
    if "gpu" in df:
        print(f"\nhardware:\n{df.groupby(['gpu', 'dtype']).size().to_string()}")

    # CC BY requires attribution, and normalisation strips image metadata.
    shutil.copy(ATTRIB, OUT_ROOT / "attribution.csv")
    att = pd.read_csv(OUT_ROOT / "attribution.csv")
    blank = att[att[["Author", "OriginalURL"]].isna().any(axis=1)]
    assert blank.empty, f"{len(blank)} attribution rows have blanks (§5.4)"
    print(f"\nattribution: {len(att)} rows, no blanks")

---

## The 2am playbook

**Retryable — re-run section 8 as-is; `_done_ids` resumes and nothing is
regenerated:**

* `CUDA out of memory` → that family needs offload. Check its `ModelSpec`:
  `offload_mode="model"` keeps one component resident, `"sequential"` one
  submodule. Do **not** quantise to fit — `docs/03` §3, a 4-bit model's traces
  are partly the compute budget's and this corpus exists to isolate the
  generator.
* `ReadTimeout` / `ConnectionError` → the Hub. Re-run.
* A worker dies but others continue → re-run; it picks up its own `--rows-dir`.

**Fatal — re-running burns GPU for nothing:**

* `jpegtran is not on PATH` → `apt-get install -y libjpeg-turbo-progs`.
* `N of M reals already have a fake under a different family` →
  `used_elsewhere` did its job. A suite ran on a shard another suite owns.
  Do not force it; move to a free shard.
* `publishes license=… registry claims …` → a model was relicensed upstream.
  That is `check_licence` working. Re-audit before generating anything with it;
  the corpus's whole licence position is the registry being true.
* `pipeline returned (w, h), smaller than the requested` → `size_multiple` is
  wrong for that model. Declare it; do not resize the output. A resample leaves
  the spectral signature `docs/resolution_shortcut.md` measured, on the
  generated class only.
* `no caption; refusing to generate on an empty prompt` at a high rate → the
  caption merge lost rows. Re-run section 4's merge; do not let generation
  caption on the fly with four workers.
* `X/Y failed, above 5%` → `run_family` stopping rather than spending hours
  producing nothing. Read `stats.json`'s `reasons`.

**And never, whatever the error:**

* Do not change `--seed` or any suite's shares. The strata pattern is a
  function of the share dict; a different one re-deals every real in the block.
* Do not re-grid to `--n-shards 4`. The boundary lands at 13,656, inside shard
  1's block.
* Do not point two workers at one `--rows-dir`, or at one captions path being
  written.
* Do not skip section 9 because the smoke images looked fine. Sharpness is the
  confound this corpus exists to fight and it is invisible by eye.